#### Libraries and MLFLOw setup

In [1]:
import dspy
from pydantic import BaseModel, Field
from typing import List, Annotated
import pandas as pd
import mlflow

# Tell MLflow about the server URI.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
# Create a unique name for your experiment.
mlflow.set_experiment("DSPy")
mlflow.dspy.autolog()

#### Setup LLM and Data

In [2]:
bedrock_model_names = [
    "us.anthropic.claude-3-haiku-20240307-v1:0",
    "us.anthropic.claude-3-opus-20240229-v1:0",
    "us.anthropic.claude-3-sonnet-20240229-v1:0",
    "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "us.anthropic.claude-3-5-sonnet-20240620-v1:0",
    "us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    "us.anthropic.claude-opus-4-20250514-v1:0",
    "us.anthropic.claude-sonnet-4-20250514-v1:0",
    "us.deepseek.r1-v1:0",
    "us.meta.llama4-maverick-17b-instruct-v1:0",
    "us.meta.llama4-scout-17b-instruct-v1:0",
    "us.meta.llama3-1-70b-instruct-v1:0",
    "us.meta.llama3-1-8b-instruct-v1:0",
    "us.meta.llama3-2-11b-instruct-v1:0",
    "us.meta.llama3-2-1b-instruct-v1:0",
    "us.meta.llama3-2-3b-instruct-v1:0",
    "us.meta.llama3-2-90b-instruct-v1:0",
    "us.meta.llama3-3-70b-instruct-v1:0",
    "us.mistral.pixtral-large-2502-v1:0",
    "us.amazon.nova-lite-v1:0",
    "us.amazon.nova-micro-v1:0",
    "us.amazon.nova-premier-v1:0",
    "us.amazon.nova-pro-v1:0",
]


lm = dspy.LM(
    model="bedrock/us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    max_tokens=32000,
    temperature=1.0,
    # top_p=0.9,
    # top_k=250,
    thinking={
        "type": "enabled",
        "budget_tokens": 4096,
    }
    # stop_sequences=["\n\n"],
)

dspy.settings.configure(lm=lm)

In [3]:
email_df = pd.read_csv("../data/emails.csv")
eval_df = pd.read_csv("../data/evaluations.csv")

In [4]:
reviewed_emails = (
    email_df.set_index("id")
    .join(
        other=eval_df.drop_duplicates(subset=["email_id"])
        .rename(columns={"id": "eval_id", "email_id": "id"})
        .set_index("id"),
        on="id",
    )
    .dropna(axis=0)
)

#### CLaude Converter

```python
"""
Simple utility for converting JSON prompt structures to Anthropic Claude Messages Format.
This module provides easy-to-use functions for converting prompt JSON files to the format
required by the Anthropic Claude API.
"""

In [5]:


import json
from typing import Dict, List, Any, Union

def json_to_anthropic_messages(json_data: Union[Dict, str]) -> List[Dict[str, str]]:
    """
    Convert JSON prompt data to Anthropic Claude Messages Format.
    
    Args:
        json_data: Either a dictionary containing the prompt data or a file path to a JSON file
        
    Returns:
        List of message objects in Anthropic Messages Format
        
    Example:
        # From dictionary
        messages = json_to_anthropic_messages(prompt_dict)
        
        # From file path
        messages = json_to_anthropic_messages("prompt.json")
    """
    if isinstance(json_data, str):
        # Assume it's a file path
        with open(json_data, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
    
    return _convert_to_messages(json_data)

def _convert_to_messages(data: Dict[str, Any]) -> List[Dict[str, str]]:
    """Internal function to convert data structure to messages."""
    messages = []
    
    # System message
    if "system_prompt" in data:
        system_content = _build_system_content(data["system_prompt"])
        messages.append({
            "role": "system",
            "content": system_content
        })
    
    # User message
    if "user_prompt" in data:
        user_content = _build_user_content(data["user_prompt"])
        messages.append({
            "role": "user",
            "content": user_content
        })
    
    return messages

def _build_system_content(system_prompt: Dict[str, Any]) -> str:
    """Build system message content."""
    parts = []
    
    if "role" in system_prompt:
        parts.append(f"Role: {system_prompt['role']}")
    
    if "objective" in system_prompt:
        parts.append(f"Objective: {system_prompt['objective']}")
    
    if "model_instructions" in system_prompt:
        parts.append("\n## Model Instructions:")
        for instruction in system_prompt["model_instructions"]:
            if "point" in instruction:
                parts.append(f"• {instruction['point']}")
    
    if "response_instructions" in system_prompt:
        parts.append("\n## Response Instructions:")
        for instruction in system_prompt["response_instructions"]:
            if "point" in instruction:
                parts.append(f"• {instruction['point']}")
    
    return "\n".join(parts)

def _build_user_content(user_prompt: Dict[str, Any]) -> str:
    """Build user message content."""
    parts = []
    
    # Positive scenarios
    if "positive_scenarios" in user_prompt:
        parts.append("\n## POSITIVE SCENARIOS (OUT_OF_SCOPE):")
        for scenario in user_prompt["positive_scenarios"]:
            if "domain" in scenario:
                parts.append(f"\n{scenario['domain']}:")
            
            if "categories" in scenario:
                for category in scenario["categories"]:
                    if "type" in category:
                        parts.append(f"  {category['type']}:")
                    
                    if "info" in category:
                        for info in category["info"]:
                            if "point" in info:
                                parts.append(f"    • {info['point']}")
    
    # Analysis instructions
    if "analysis_instructions" in user_prompt:
        analysis = user_prompt["analysis_instructions"]
        
        if "instructions" in analysis:
            parts.append("\n## ANALYSIS INSTRUCTIONS:")
            for instruction in analysis["instructions"]:
                if "point" in instruction:
                    parts.append(f"• {instruction['point']}")
        
        # Confidence score guidelines
        if "confidence_score_assignment" in analysis:
            confidence = analysis["confidence_score_assignment"]
            
            if "positve_classication_score_guidelines" in confidence:
                parts.append("\n###  POSITIVE CLASSIFICATION SCORE GUIDELINES:")
                for guideline in confidence["positve_classication_score_guidelines"]:
                    if "point" in guideline:
                        parts.append(f"• {guideline['point']}")
            
            if "negative_classication_score_guidelines" in confidence:
                parts.append("\n###  NEGATIVE CLASSIFICATION SCORE GUIDELINES:")
                for guideline in confidence["negative_classication_score_guidelines"]:
                    if "point" in guideline:
                        parts.append(f"• {guideline['point']}")
    
    # Output instructions
    if "output_instructions" in user_prompt:
        parts.append("\n## OUTPUT INSTRUCTIONS:")
        for instruction in user_prompt["output_instructions"]:
            if "point" in instruction:
                parts.append(f"• {instruction['point']}")
    
    # Examples
    if "examples" in user_prompt:
        parts.append("\n## EXAMPLES:")
        for i, example in enumerate(user_prompt["examples"], 1):
            parts.append(f"\nExample {i}:")
            parts.append("\n Input: \n")
            if "prompt_input" in example:
                input_data = example["prompt_input"]
                if "subject" in input_data:
                    parts.append(f"Subject: {input_data['subject']}")
                if "body" in input_data:
                    body_lines = input_data['body'].split('\n')
                    body_lines = [line.strip() for line in body_lines if len(line.strip()) > 0]
                    parts.append(f"Body: {'\n'.join(body_lines)}")
            parts.append("\n Output: \n")
            if "prompt_output" in example:
                output_data = example["prompt_output"]
                if "confidence_score" in output_data:
                    parts.append(f"Confidence Score: {output_data['confidence_score']}")
                if "classification_rationale" in output_data:
                    parts.append(f"Classification Rationale: {output_data['classification_rationale']}")
                if "classification_label" in output_data:
                    parts.append(f"Classification Label: {output_data['classification_label']}")
            
            parts.append("\n\n"+'-'*100 +"\n")
    
    return "\n".join(parts)

# Convenience function for direct file conversion
def convert_file(input_file: str, output_file: str = None) -> List[Dict[str, str]]:
    """
    Convert a JSON file to Anthropic Messages Format and optionally save the result.
    
    Args:
        input_file: Path to the input JSON file
        output_file: Optional path to save the converted messages (as JSON)
        
    Returns:
        List of message objects in Anthropic Messages Format
    """
    messages = json_to_anthropic_messages(input_file)
    
    if output_file:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(messages, f, indent=2, ensure_ascii=False)
        print(f"✅ Converted messages saved to: {output_file}")
    
    return messages

# if __name__ == "__main__":
#     # Simple command-line usage
#     import sys
    
#     if len(sys.argv) < 2:
#         print("Usage: python anthropic_converter.py <input_file> [output_file]")
#         sys.exit(1)
    
#     input_file = sys.argv[1]
#     output_file = sys.argv[2] if len(sys.argv) > 2 else None
    
#     try:
#         messages = convert_file(input_file, output_file)
#         print(f"✅ Successfully converted {len(messages)} messages")
#     except Exception as e:
#         print(f"❌ Error: {e}")
#         sys.exit(1) 

### Out Of Scope

#### Data Collection

In [6]:
pure_out_of_scope = reviewed_emails[reviewed_emails["classes"] == "{OUT_OF_SCOPE}"]


# dataset = list(zip(pure_out_of_scope[['subject','body']].to_dict(orient="list")['subject'], pure_out_of_scope[['subject','body']].to_dict(orient="list")['body']))
# dataset = []

sample = pure_out_of_scope.iloc[0].to_dict()
pure_out_of_scope

,body,subject,classes,reasons,eval_id,user_id,new_class,new_reason,timestamp
id,,,,,,,,,
1,From: Microsoft Outlook (/o=ExchangeLabs/ou=Ex...,Undeliverable: Seabourn Availability- Mr Seetu...,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",9.0,1.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",2025-07-25 17:36:54.890825
2,From: postmaster@gac.com (postmaster@gac.com)\...,[EXTERNAL] Your message couldn't be delivered,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",14.0,3.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",2025-07-25 20:36:57.926714
4,"From: Crew Hotel, Fleet (PCL) (/o=ExchangeLabs...",Re: [EXTERNAL] JOINERS / LEAVERS FOR SEABOURN ...,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",16.0,3.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",2025-07-25 20:43:39.969958
5,From: SBNHotel Joiners (/o=ExchangeLabs/ou=Exc...,SBN | Australian Maritime Crew Visa Application,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""External vendor providing vi...",17.0,3.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""External vendor providing vi...",2025-07-25 20:44:21.790404
7,From: Sirin Arabaci (cruise@paralianshipping.c...,[EXTERNAL] NO SHOW - OKTB Request - SEABOURN E...,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""External vendor communicatio...",33.0,1.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""External vendor communicatio...",2025-07-28 16:40:49.488129
...,...,...,...,...,...,...,...,...,...
744,From: Microsoft Outlook (/o=ExchangeLabs/ou=Ex...,Undeliverable: GUZMAN MADRIGAL Luis Miguel (Co...,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",5.0,1.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",2025-07-25 17:36:54.890819
798,From: Microsoft Outlook (/o=ExchangeLabs/ou=Ex...,Undeliverable: Luggage Lost - Barbara Andrade ...,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",12.0,1.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",2025-07-25 17:36:54.890829
832,From: Kim Hills-Harrop (/o=ExchangeLabs/ou=Exc...,FW: [EXTERNAL] Morozov Mykyta bartender,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",119.0,3.0,{OUT_OF_SCOPE},"{""OUT_OF_SCOPE"": ""Unable to classify the email""}",2025-07-28 18:40:07.34251


#### Rationale Finder

In [7]:
system_prompt = """<ROLE> You are an expert email classification agent specializing in identifying Out of Scope communications from email communications. Your task is to analyze email content and determine if it constitutes external vendor communications, recruitment agency solicitations, spam/phishing attempts, or other communications that should not be processed by internal systems. </ROLE> <OBJECTIVE> Classify whether the given email falls under the category of "Out of Scope" based on the presence of external vendors, recruitment agencies, spam/phishing attempts, marketing communications, or other non-internal business communications that should be filtered out. </OBJECTIVE> <MODEL_INSTRUCTIONS> - Out of scope communications should be filtered out from internal processing - Spam/phishing attempts require immediate blocking and security review - Vendor communications should be routed to appropriate procurement teams - Recruitment communications should be handled by HR/recruitment teams - Internal employee communications should always be processed - External domains are strong indicators of out of scope communications - Marketing language and promotional offers indicate external communications - Confidence scores help determine filtering accuracy - Escalation flags help route security threats appropriately </MODEL_INSTRUCTIONS> <RESPONSE_INSTRUCTIONS> Respond ONLY with the XML classification result </RESPONSE_INSTRUCTIONS>"""

user_prompt = """<POSITIVE_KEYWORDS> - vendor - supplier - contractor - external provider - recruitment - headhunting - job placement - staffing - spam - phishing - suspicious - scam - fraud - marketing - sales pitch - promotional - advertisement - external - third party - outside company - job application - resume - CV - candidate - service provider - consultant - agency - unsubscribe - marketing email - newsletter - special offer - limited time - act now - free consultation - exclusive pricing - executive search - confidential opportunity - lottery - prize - winner - claim now - account suspended - click here - external domain - non-company email </POSITIVE_KEYWORDS> <POSITIVE_SCENARIOS> 1. **Vendor and Supplier Communications**: - Equipment and service providers - Sales pitches and promotional offers - Pricing inquiries and proposals - External service offerings - Vendor catalogs and brochures 2. **Recruitment and Staffing Agencies**: - Job placement services - Executive search firms - Candidate database offerings - Staffing solutions - Recruitment partnerships 3. **Spam and Phishing Attempts**: - Suspicious account verification requests - Lottery and prize notifications - - Urgent action required messages - - Requests for personal information - - Suspicious links and attachments 4. **Marketing and Advertising**: - Promotional emails and newsletters - Special offers and discounts - Industry newsletters - Marketing campaigns - Subscription services 5. **External Business Inquiries**: - Job applications from external candidates - Partnership proposals - Business development inquiries - External service requests - Unrelated business opportunities </OUT_OF_SCOPE_SCENARIOS> <CONFIDENCE_SCORE_GUIDELINES> **Contextual Meaning of Confidence Score:** The confidence score reflects the model's certainty about the classification decision, regardless of whether the result is positive or negative. **When is_positive=true (Email classified as Out of Scope):** - **High Confidence (0.9-1.0)**: Strong certainty that email is out of scope - Clear external vendor or spam indicators - Suspicious phishing attempts - Marketing language with external domains - Complete external business context - **Medium Confidence (0.7-0.89)**: Moderate certainty that email is out of scope - Recruitment agency communications - Marketing emails and newsletters - External service providers - Most information indicates external origin - **Low Confidence (0.5-0.69)**: Weak certainty that email is out of scope - Some external indicators - Unclear business context - Mixed internal/external elements - Requires further analysis **When is_positive=false (Email classified as In Scope):** - **High Confidence (0.9-1.0)**: Strong certainty that email is in scope - Clear internal communication - Employee requests and inquiries - Company operational matters - No external indicators - **Medium Confidence (0.7-0.89)**: Moderate certainty that email is in scope - Internal employee communications - Assignment or emergency requests - Work-related inquiries - Company domain indicators - **Low Confidence (0.5-0.69)**: Weak certainty that email is in scope - Some ambiguous internal/external indicators - Context could be interpreted multiple ways - Unclear business context - Borderline cases requiring human judgment **Interpretability:** - High confidence (0.9-1.0) = Strong certainty in the classification decision (positive or negative) - Medium confidence (0.7-0.89) = Moderate certainty in the classification decision - Low confidence (0.5-0.69) = Weak certainty in the classification decision - Very low confidence (0.0-0.49) = Very weak certainty, likely requiring human review </CONFIDENCE_SCORE_GUIDELINES> <ANALYSIS_INSTRUCTIONS> 1. **Sender Analysis**: Check email domain and sender information 2. **Content Analysis**: Evaluate message content and intent 3. **Language Assessment**: Identify marketing, sales, or external business language 4. **Actionability**: Determine if communication requires internal processing 5. **Risk Assessment**: Evaluate potential spam/phishing indicators </ANALYSIS_INSTRUCTIONS> <OUTPUT_INSTRUCTIONS> You must respond with ONLY the XML classification result. Do not include any other text, explanations, or formatting outside the XML tags. And Strictly follow the XML structure provided below. <CLASSIFICATION_OUTPUT> <classification> <is_positive>true/false</is_positive> <confidence_score>0.0-1.0</confidence_score> <keywords_found> <keyword>keyword1</keyword> <keyword>keyword2</keyword> </keywords_found> <rationale>Brief explanation of classification (max 20 words)</rationale> <flag_for_review>true/false</flag_for_review> </classification> </CLASSIFICATION_OUTPUT> **Flag for Review Criteria:** - Set to true when confidence_score < 0.7 (low certainty in classification decision) - Set to true when confidence_score < 0.5 (very low certainty, requires human review) - Set to true when risk_level is "high" (potential security threats) - Set to true when sender_analysis is "suspicious" or "unclear" (security concerns) - Set to true when there are mixed internal/external indicators (ambiguous scope) - Set to true when classification requires human judgment - Set to true when borderline cases exist (confidence 0.5-0.69) </OUTPUT_INSTRUCTIONS> <EXAMPLES> <EXAMPLE_1> Example 1: Email: "Special Offer - 50% Discount on Equipment" From: sales@vendor.com Response: <classification> <is_positive>true</is_positive> <confidence_score>0.95</confidence_score> <sub_classification>vendor</sub_classification> <risk_level>low</risk_level> <keywords_found> <keyword>special offer</keyword> <keyword>discount</keyword> <keyword>equipment</keyword> </keywords_found> <sender_analysis>external</sender_analysis> <missing_information> <missing_detail>specific equipment</missing_detail> <missing_detail>pricing details</missing_detail> </missing_information> <rationale>Clear vendor sales communication with promotional language</rationale> <flag_for_review>false</flag_for_review> <suggested_action>Filter out - no internal processing required</suggested_action> <escalation_needed>false</escalation_needed> </classification> </EXAMPLE_1> <EXAMPLE_2> Example 2: Email: "URGENT: Your Account Has Been Suspended" From: security@bank-verify.com Response: <classification> <is_out_of_scope>true</is_out_of_scope> <is_in_scope>false</is_in_scope> <confidence_score>0.98</confidence_score> <sub_classification>spam_phishing</sub_classification> <risk_level>high</risk_level> <keywords_found> <keyword>urgent</keyword> <keyword>account suspended</keyword> <keyword>verify</keyword> </keywords_found> <sender_analysis>suspicious</sender_analysis> <missing_information> <missing_detail>specific account</missing_detail> <missing_detail>verification process</missing_detail> </missing_information> <rationale>Suspicious phishing attempt with urgent action required</rationale> <flag_for_review>true</flag_for_review> <suggested_action>Block and flag as potential security threat</suggested_action> <escalation_needed>true</escalation_needed> </classification> </EXAMPLE_2> <EXAMPLE_3> Example 3: Email: "Request for Assignment Change" From: employee@company.com Response: <classification> <is_out_of_scope>false</is_out_of_scope> <is_in_scope>true</is_in_scope> <confidence_score>0.95</confidence_score> <sub_classification>in_scope</sub_classification> <risk_level>none</risk_level> <keywords_found> <keyword>assignment change</keyword> <keyword>request</keyword> </keywords_found> <sender_analysis>internal</sender_analysis> <missing_information> <missing_detail>specific assignment</missing_detail> <missing_detail>reason for change</missing_detail> </missing_information> <rationale>Internal employee request for assignment change</rationale> <flag_for_review>false</flag_for_review> <suggested_action>Process through assignment change workflow</suggested_action> <escalation_needed>false</escalation_needed> </classification> </EXAMPLE_3> </EXAMPLES> <EMAIL_CONTENT> Analyze the following email content and respond with ONLY the XML classification: {{body}} {{subject}} </EMAIL_CONTENT>"""

#### Prompt Creator

In [8]:
from pydantic import BaseModel, Field
from typing import List, Annotated


class BulletPoint(BaseModel):
    point: Annotated[
        str,
        Field(
            ...,
            description="A single concise statement detailing any functionality, scenario or information about a topic or use case",
        ),
    ]


class SystemPrompt(BaseModel):
    role: Annotated[
        str,
        Field(..., description="The role of the agent described in 2 to 3 sentences"),
    ]
    objective: Annotated[
        str,
        Field(
            ..., description="The objective of the agent described in 4 to 5 sentences"
        ),
    ]
    model_instructions: Annotated[
        list[BulletPoint],
        Field(..., description="The model instructions limited to 5 points"),
    ]
    response_instructions: Annotated[
        list[BulletPoint],
        Field(..., description="The response instructions limited to 4 points"),
    ]


class PromptInput(BaseModel):
    body: Annotated[str, Field(..., description="The email body")]
    subject: Annotated[str, Field(..., description="The email subject")]


class DomainCategory(BaseModel):
    type: Annotated[
        str,
        Field(
            ...,
            description="Defines the specific business function or the primary nature of the risk associated with the email. It serves as the main classifier that refines the broader Domain, grouping communications by their core purpose.",
        ),
    ]
    info: Annotated[
        list[BulletPoint],
        Field(
            ...,
            description="Offers a concise definitions and concrete examples of the scenarios that fall under the specified Type",
        ),
    ]


class PositiveScenario(BaseModel):
    domain: Annotated[
        str,
        Field(
            ...,
            description="The Domain is the highest-level classifier, grouping emails by their fundamental purpose or strategic importance, such as commercial, business development, or information security risk.",
        ),
    ]
    categories: Annotated[
        list[DomainCategory],
        Field(
            ...,
            description="A list of categories that further specify the business function or risk type within the domain, each with definitions and examples.",
        ),
    ]


class ConfidenceScoreGuidelines(BaseModel):
    positve_classication_score_guidelines: Annotated[
        list[BulletPoint],
        Field(
            ...,
            description="The positive classification score guidelines used while assigning the confidence score in case of positive classification",
        ),
    ]
    negative_classication_score_guidelines: Annotated[
        list[BulletPoint],
        Field(
            ...,
            description="The negative classification score guidelines used while assigning the confidence score in case of negative classification",
        ),
    ]


class AnalysisInstructions(BaseModel):
    instructions: Annotated[
        list[BulletPoint],
        Field(
            ...,
            description="The instructions that needs to be followed for the analysis of the email",
        ),
    ]
    confidence_score_assignment: Annotated[
        ConfidenceScoreGuidelines,
        Field(..., description="The confidence score assignment"),
    ]


class PromptOutput(BaseModel):
    confidence_score: Annotated[float, Field(..., description="The confidence score")]
    classification_label: Annotated[
        str, Field(..., description="The classification label")
    ]
    classification_rationale: Annotated[
        str, Field(..., description="The classification rationale")
    ]


class Sample(BaseModel):
    prompt_input: PromptInput = dspy.InputField(
        description="The sample data that the prompt needs to process"
    )
    prompt_output: PromptOutput = dspy.InputField(
        description="The output of the prompt"
    )


class UserPrompt(BaseModel):
    positive_scenarios: Annotated[
        list[PositiveScenario],
        Field(..., description="List of the Positive Scenarios Identified"),
    ]
    analysis_instructions: Annotated[
        AnalysisInstructions, Field(..., description="The analysis instructions")
    ]
    output_instructions: Annotated[
        list[BulletPoint],
        Field(
            ...,
            description="The output instructions that are needed to output the object as JSON object",
        ),
    ]
    examples: Annotated[list[Sample], Field(..., description="The examples")]


class Prompt(BaseModel):
    system_prompt: Annotated[SystemPrompt, Field(..., description="The system prompt")]
    user_prompt: Annotated[UserPrompt, Field(..., description="The user prompt")]


In [9]:
import json

with open('out_of_scope_samples.json', 'r') as f:
    data = json.load(f)


sample = data[2]
sample

{'body': 'From: Crew Hotel, Fleet (PCL) (/o=ExchangeLabs/ou=Exchange Administrative Group (FYDIBOHF23SPDLT)/cn=Recipients/cn=c4a978c08c844cca946bbe387cd59e6d-Fleet Crew)\n\nTo: Crew Services (crewservices@platinumportagency.com)\n\nGood Day,\r\n\r\nPlease see ERS details\r\n\r\n\r\n\r\n\r\n\r\nBest Regards,\r\n\r\nHaley\r\n\r\n\r\n\r\nHolland America Group | Serving Princess Cruises, Holland America Line, Seabourn, and P&O Australia \r\n\r\n24305 Town Center Drive | Santa Clarita, CA 91355   \r\n\r\ncrewhotel@hagroup.com <mailto:crewhotel@hagroup.com>            \r\n\r\n24Hr Emergency Line: +1 206-262-5800 (Prompt 1 for crew) \r\n\r\nFleet Crew Hotel’s Regular Operation Hours: 8:00am – 5:00pm PST Monday - Friday\r\n\r\n\r\n________________________________\r\n\r\nFrom: Crew Hotel, Fleet (PCL) <crewhotel@hagroup.com>\r\nSent: Thursday, March 13, 2025 10:48 AM\r\nTo: Crew Services <crewservices@platinumportagency.com>\r\nCc: SBNHotel Joiners <sbnhoteljoiners@seabourn.com>\r\nSubject: Re: 

In [10]:
from typing import Optional

class PromptGeneratorSignature(dspy.Signature):
    input_samples: list[Sample] = dspy.InputField(description="The input samples that the prompt needs to process with their corresponding outputs")
    background_info: Optional[str] = dspy.InputField(description="The background information about the prompt")
    prompt: Prompt = dspy.OutputField(description="The prompt that needs to be generated follow the best practices of prompt engineering while developing the prompt.")

class PromptGenerator(dspy.Module):
    def __init__(self):
        self.prompt_generator_signature = dspy.ReAct(PromptGeneratorSignature, tools=[])
        self.hint = "The prompt generator is used to generate a prompt for the model. The prompt should be generated follow the best practices of prompt engineering while developing the prompt. The task is to generate a prompt that can be used to classify the email into OUT_OF_SCOPE or IN_SCOPE."
    
    def forward(self, input_samples: list[Sample]) -> Prompt:
        return self.prompt_generator_signature(
            input_samples=input_samples,
            background_info=self.hint
        )


pg = PromptGenerator()


# result_prompt = pg(pi, po)

In [18]:
import random

random.shuffle(data)

test_samples = [
    Sample(
        prompt_input=PromptInput(
            body=sample['body'][5000:],
            subject=sample['subject']
        ),
        prompt_output=PromptOutput(
            classification_label="OUT_OF_SCOPE",
            confidence_score=round(random.uniform(0.8, 1.0),2),
            classification_rationale=sample['better_rationale']
        )
    )
    for sample in data[:30]
]

result_prompt = pg(test_samples)

# print(result_prompt.prompt.model_dump_json(indent=4))

Trace(trace_id=tr-60c7280ac6c3a2b6ec317444b1783daf)

In [15]:
# print(result_prompt.prompt.model_dump_json(indent=4))

with open('generated_prompt.json', 'w') as f:
    f.write(result_prompt.prompt.model_dump_json(indent=4))

In [16]:
out = convert_file('generated_prompt.json', 'output_messages.json')

✅ Converted messages saved to: output_messages.json


In [17]:
with open("output_messages.json", "r") as f:
    messages = json.load(f)

with open('prompt.md', 'w') as f:
    for msg in messages:
        print(msg['content'])
        f.write(msg['content'])

Role: You are an expert email classification agent specializing in distinguishing between emails that should be processed by internal systems (IN_SCOPE) and those that should be routed elsewhere (OUT_OF_SCOPE). Your task is to analyze each email's content and subject line to determine the appropriate classification.
Objective: You will analyze incoming emails to determine if they should be classified as IN_SCOPE or OUT_OF_SCOPE based on specific criteria. IN_SCOPE emails represent internal business communications that should be processed through standard company systems. OUT_OF_SCOPE emails involve external vendors, technical notifications, or communications requiring specialized routing. Your goal is to make accurate classifications with appropriate confidence levels, providing clear rationales for each decision.

## Model Instructions:
• Examine both the email subject and body content carefully before making classification decisions
• Consider sender and recipient domains as importan

samples, -> Prompt


Prompt run inference on -> wider dataset
each inference -> feedback

track accuracy of the prompt

using the feedback -> new prompt

early stopping threshold
tol=0.03 stop

 **Critical**
custom metric where a prompt is given as output and its efficiency is calculated against a given dataset.


If critical is solved then we can use dspy optimizers which will make it more robust.